# Phase 3: Optuna HPO + Ensemble (LGBM + XGB + RF)

Mejoras sobre Fase 2:
1. **Optuna** — búsqueda de hiperparámetros óptimos para LightGBM (50 trials)
2. **XGBoost** — segundo modelo con su propio tuning
3. **RandomForest** — tercer modelo para diversidad
4. **Ensemble** — promedio ponderado por OOF AUC de los 3 modelos

Métrica objetivo: AUC

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
import warnings
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED     = 42
N_FOLDS  = 5
TE_ALPHA = 10
N_TRIALS = 50  # trials de Optuna por modelo

INPUT_PATH  = Path('/Users/haroldlagares/Downloads/competition/input')
RESULTS_DIR = Path('/Users/haroldlagares/Downloads/competition/results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f'LightGBM {lgb.__version__} | XGBoost {xgb.__version__}')

## 2. Load & Preprocessing (idéntico a Fase 2)

In [ ]:
train_raw = pd.read_csv(INPUT_PATH / 'train.csv')
test_raw  = pd.read_csv(INPUT_PATH / 'test.csv')

NULL_COLS    = ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
PERF_COLS    = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
INVERSE_COLS = ['Sprint_40yd', 'Agility_3cone', 'Shuttle']
TE_COLS      = ['School', 'Position', 'Position_Type']

train = train_raw.copy()
test  = test_raw.copy()

# Missing flags
for col in NULL_COLS:
    train[f'missing_{col}'] = train[col].isna().astype(np.int8)
    test[f'missing_{col}']  = test[col].isna().astype(np.int8)

# Group imputation por Position_Type
group_medians = train.groupby('Position_Type')[NULL_COLS].median()
global_fallback = group_medians.median()
for col in NULL_COLS:
    for df in [train, test]:
        mask = df[col].isna()
        if mask.any():
            vals = df.loc[mask, 'Position_Type'].map(group_medians[col]).fillna(global_fallback[col])
            df.loc[mask, col] = vals

# BMI
for df in [train, test]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

# Z-scores por Position
pos_stats = {}
for col in PERF_COLS:
    pos_stats[col] = {
        'mean': train.groupby('Position')[col].mean(),
        'std':  train.groupby('Position')[col].std().replace(0, np.nan).fillna(1),
    }

for df in [train, test]:
    for col in PERF_COLS:
        mean_map = df['Position'].map(pos_stats[col]['mean'])
        std_map  = df['Position'].map(pos_stats[col]['std']).replace(0, np.nan).fillna(1)
        z = (df[col] - mean_map) / std_map
        if col in INVERSE_COLS:
            z = -z
        df[f'z_{col}'] = z

# Overall athleticism
z_cols = [f'z_{c}' for c in PERF_COLS]
for df in [train, test]:
    df['overall_athleticism'] = df[z_cols].mean(axis=1)

# Label encode Player_Type
le = LabelEncoder()
le.fit(pd.concat([train['Player_Type'], test['Player_Type']], ignore_index=True).astype(str))
train['Player_Type'] = le.transform(train['Player_Type'].astype(str))
test['Player_Type']  = le.transform(test['Player_Type'].astype(str))

# Target encoding estático
def target_encode_static(train_df, apply_df, col, target='Drafted', alpha=10):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['te'] = (stats['mean'] * stats['count'] + global_mean * alpha) / (stats['count'] + alpha)
    return apply_df[col].map(stats['te']).fillna(global_mean)

for col in TE_COLS:
    train[f'te_{col}'] = target_encode_static(train_raw, train, col, 'Drafted', TE_ALPHA)
    test[f'te_{col}']  = target_encode_static(train_raw, test,  col, 'Drafted', TE_ALPHA)

FEATURE_COLS = [
    'Year', 'Age', 'Height', 'Weight',
    'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle',
    'Player_Type',
    'missing_Age', 'missing_Sprint_40yd', 'missing_Vertical_Jump', 'missing_Bench_Press_Reps',
    'missing_Broad_Jump', 'missing_Agility_3cone', 'missing_Shuttle',
    'BMI',
    'z_Sprint_40yd', 'z_Vertical_Jump', 'z_Bench_Press_Reps', 'z_Broad_Jump', 'z_Agility_3cone', 'z_Shuttle',
    'overall_athleticism',
    'te_School', 'te_Position', 'te_Position_Type',
]

TARGET_COL = 'Drafted'
y = train[TARGET_COL].values
X = train[FEATURE_COLS].values
X_test = test[FEATURE_COLS].values

print(f'Features: {len(FEATURE_COLS)} | Train: {X.shape} | Test: {X_test.shape}')
print('Preprocessing completo.')

## 3. Optuna — Tuning de LightGBM

Optimizamos sobre OOF AUC con 50 trials. El espacio de búsqueda cubre los hiperparámetros más influyentes de LGBM.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def lgbm_objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'auc',
        'verbose':           -1,
        'seed':              SEED,
        'learning_rate':     trial.suggest_float('learning_rate',     0.01,  0.1,  log=True),
        'num_leaves':        trial.suggest_int(  'num_leaves',        16,    256),
        'max_depth':         trial.suggest_int(  'max_depth',         3,     12),
        'min_child_samples': trial.suggest_int(  'min_child_samples', 5,     100),
        'feature_fraction':  trial.suggest_float('feature_fraction',  0.4,   1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction',  0.4,   1.0),
        'bagging_freq':      trial.suggest_int(  'bagging_freq',      1,     10),
        'reg_alpha':         trial.suggest_float('reg_alpha',         1e-4,  10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda',        1e-4,  10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain',    0.0,   1.0),
    }

    oof = np.zeros(len(X))
    for train_idx, val_idx in skf.split(X, y):
        dtrain = lgb.Dataset(X[train_idx], label=y[train_idx], feature_name=FEATURE_COLS)
        dval   = lgb.Dataset(X[val_idx],   label=y[val_idx],   feature_name=FEATURE_COLS, reference=dtrain)
        model  = lgb.train(
            params, dtrain,
            num_boost_round=1000,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(-1),
            ],
        )
        oof[val_idx] = model.predict(X[val_idx])
    return roc_auc_score(y, oof)


print(f'Iniciando Optuna para LightGBM ({N_TRIALS} trials)...')
lgbm_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_lgbm_params = lgbm_study.best_params
best_lgbm_params.update({'objective': 'binary', 'metric': 'auc', 'verbose': -1, 'seed': SEED})

print(f'\nMejor OOF AUC LGBM: {lgbm_study.best_value:.5f}')
print('Mejores params:')
for k, v in lgbm_study.best_params.items():
    print(f'  {k}: {v}')

## 4. Train LightGBM con Best Params (OOF + Test)

In [ ]:
lgbm_oof   = np.zeros(len(X))
lgbm_test  = np.zeros(len(X_test))
lgbm_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    dtrain = lgb.Dataset(X[train_idx], label=y[train_idx], feature_name=FEATURE_COLS)
    dval   = lgb.Dataset(X[val_idx],   label=y[val_idx],   feature_name=FEATURE_COLS, reference=dtrain)
    model  = lgb.train(
        best_lgbm_params, dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    val_pred = model.predict(X[val_idx])
    lgbm_oof[val_idx] = val_pred
    lgbm_test += model.predict(X_test) / N_FOLDS
    fold_auc = roc_auc_score(y[val_idx], val_pred)
    lgbm_scores.append(fold_auc)
    print(f'Fold {fold} | AUC: {fold_auc:.5f} | Best iter: {model.best_iteration}')

lgbm_oof_auc = roc_auc_score(y, lgbm_oof)
print(f'\nLGBM OOF AUC: {lgbm_oof_auc:.5f}  (±{np.std(lgbm_scores):.5f})')

## 5. Optuna — Tuning de XGBoost

In [ ]:
def xgb_objective(trial):
    params = {
        'objective':        'binary:logistic',
        'eval_metric':      'auc',
        'verbosity':        0,
        'seed':             SEED,
        'learning_rate':    trial.suggest_float('learning_rate',    0.01,  0.1,  log=True),
        'max_depth':        trial.suggest_int(  'max_depth',        3,     10),
        'min_child_weight': trial.suggest_int(  'min_child_weight', 1,     50),
        'subsample':        trial.suggest_float('subsample',        0.5,   1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4,   1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha',        1e-4,  10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda',       1e-4,  10.0, log=True),
        'gamma':            trial.suggest_float('gamma',            0.0,   5.0),
    }

    oof = np.zeros(len(X))
    for train_idx, val_idx in skf.split(X, y):
        dtrain = xgb.DMatrix(X[train_idx], label=y[train_idx], feature_names=FEATURE_COLS)
        dval   = xgb.DMatrix(X[val_idx],   label=y[val_idx],   feature_names=FEATURE_COLS)
        model  = xgb.train(
            params, dtrain,
            num_boost_round=1000,
            evals=[(dval, 'val')],
            early_stopping_rounds=50,
            verbose_eval=False,
        )
        oof[val_idx] = model.predict(dval)
    return roc_auc_score(y, oof)


print(f'Iniciando Optuna para XGBoost ({N_TRIALS} trials)...')
xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_xgb_params = xgb_study.best_params
best_xgb_params.update({'objective': 'binary:logistic', 'eval_metric': 'auc', 'verbosity': 0, 'seed': SEED})

print(f'\nMejor OOF AUC XGB: {xgb_study.best_value:.5f}')
print('Mejores params:')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')

## 6. Train XGBoost con Best Params (OOF + Test)

In [ ]:
xgb_oof   = np.zeros(len(X))
xgb_test  = np.zeros(len(X_test))
xgb_scores = []

dtest_xgb = xgb.DMatrix(X_test, feature_names=FEATURE_COLS)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    dtrain = xgb.DMatrix(X[train_idx], label=y[train_idx], feature_names=FEATURE_COLS)
    dval   = xgb.DMatrix(X[val_idx],   label=y[val_idx],   feature_names=FEATURE_COLS)
    model  = xgb.train(
        best_xgb_params, dtrain,
        num_boost_round=1000,
        evals=[(dval, 'val')],
        early_stopping_rounds=50,
        verbose_eval=False,
    )
    val_pred = model.predict(dval)
    xgb_oof[val_idx] = val_pred
    xgb_test += model.predict(dtest_xgb) / N_FOLDS
    fold_auc = roc_auc_score(y[val_idx], val_pred)
    xgb_scores.append(fold_auc)
    print(f'Fold {fold} | AUC: {fold_auc:.5f} | Best iter: {model.best_iteration}')

xgb_oof_auc = roc_auc_score(y, xgb_oof)
print(f'\nXGB OOF AUC: {xgb_oof_auc:.5f}  (±{np.std(xgb_scores):.5f})')

## 7. RandomForest (diversidad en el ensemble)

In [ ]:
rf_oof   = np.zeros(len(X))
rf_test  = np.zeros(len(X_test))
rf_scores = []

rf_params = {
    'n_estimators':   500,
    'max_depth':      None,
    'min_samples_leaf': 5,
    'max_features':   'sqrt',
    'class_weight':   'balanced',
    'random_state':   SEED,
    'n_jobs':         -1,
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    rf = RandomForestClassifier(**rf_params)
    rf.fit(X[train_idx], y[train_idx])
    val_pred = rf.predict_proba(X[val_idx])[:, 1]
    rf_oof[val_idx] = val_pred
    rf_test += rf.predict_proba(X_test)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y[val_idx], val_pred)
    rf_scores.append(fold_auc)
    print(f'Fold {fold} | AUC: {fold_auc:.5f}')

rf_oof_auc = roc_auc_score(y, rf_oof)
print(f'\nRF OOF AUC: {rf_oof_auc:.5f}  (±{np.std(rf_scores):.5f})')

## 8. Ensemble — Promedio Ponderado por OOF AUC

Peso de cada modelo proporcional a su OOF AUC. Modelos más precisos contribuyen más.

In [ ]:
model_aucs = {
    'LGBM': lgbm_oof_auc,
    'XGB':  xgb_oof_auc,
    'RF':   rf_oof_auc,
}

total_auc = sum(model_aucs.values())
weights   = {k: v / total_auc for k, v in model_aucs.items()}

print('=== Resumen de modelos ===')
for name, auc in model_aucs.items():
    print(f'  {name:<6} OOF AUC: {auc:.5f}  |  peso: {weights[name]:.4f}')

# OOF ensemble
ens_oof = (
    weights['LGBM'] * lgbm_oof +
    weights['XGB']  * xgb_oof  +
    weights['RF']   * rf_oof
)
ens_oof_auc = roc_auc_score(y, ens_oof)

# Test ensemble
ens_test = (
    weights['LGBM'] * lgbm_test +
    weights['XGB']  * xgb_test  +
    weights['RF']   * rf_test
)

print(f'\nEnsemble OOF AUC: {ens_oof_auc:.5f}')
print(f'Ganancia vs mejor modelo individual: {ens_oof_auc - max(model_aucs.values()):+.5f}')

## 9. Guardar Submission

In [ ]:
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
submission_path = RESULTS_DIR / f'submission_phase3_{timestamp}.csv'

submission = pd.read_csv(INPUT_PATH / 'sample_submission.csv')
submission['Drafted'] = ens_test
submission.to_csv(submission_path, index=False)

print(f'Submission: {submission_path}')
print(f'Ensemble OOF AUC : {ens_oof_auc:.5f}')
print(f'LGBM OOF AUC     : {lgbm_oof_auc:.5f}')
print(f'XGB  OOF AUC     : {xgb_oof_auc:.5f}')
print(f'RF   OOF AUC     : {rf_oof_auc:.5f}')
print(submission.head())